### Set up the remote kernel
If you use a remote kernel, then you need to set it up with all local code and
the corresponding required dependencies so that we can run our LLM in the Colab
while using the GPU.

In [1]:
import sys
sys.path

['/content',
 '/env/python',
 '/usr/lib/python312.zip',
 '/usr/lib/python3.12',
 '/usr/lib/python3.12/lib-dynload',
 '',
 '/usr/local/lib/python3.12/dist-packages',
 '/usr/lib/python3/dist-packages',
 '/usr/local/lib/python3.12/dist-packages/IPython/extensions',
 '/root/.ipython']

#### Clone repositories

In [2]:
!git clone https://github.com/NiccoloSacchi/assignment1-basics.git
print()
!git clone https://github.com/NiccoloSacchi/assignment2-systems.git
print()

Cloning into 'assignment1-basics'...
remote: Enumerating objects: 390, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 390 (delta 0), reused 0 (delta 0), pack-reused 389 (from 2)
Receiving objects: 100% (390/390), 14.48 MiB | 15.56 MiB/s, done.
Resolving deltas: 100% (214/214), done.

Cloning into 'assignment2-systems'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 119 (delta 4), reused 7 (delta 3), pack-reused 110 (from 2)
Receiving objects: 100% (119/119), 1.02 MiB | 5.65 MiB/s, done.
Resolving deltas: 100% (57/57), done.



In [3]:
# Install Assignment 2's dependencies:
# --system: directly into the environment the Colab kernel is already using.
# -e: in "editable" mode, if you change code in the Assignment 1 folder, the
#   changes are reflected immediately without a reinstall.
!cd assignment2-systems; uv pip install --system -e .

Using Python 3.12.12 environment at: /usr
Resolved 76 packages in 704ms                                        
Prepared 26 packages in 55.70s                                           
Uninstalled 17 packages in 828ms
Installed 26 packages in 268ms                              
 + cs336-basics==1.0.6 (from file:///content/assignment1-basics)
 + cs336-systems==1.0.5 (from file:///content/assignment2-systems)
 + einx==0.4.1
 + humanfriendly==10.0
 + jaxtyping==0.3.9
 + memory-profiler==0.61.0
 - nvidia-cublas-cu12==12.8.4.1
 + nvidia-cublas-cu12==12.4.5.8
 - nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-cupti-cu12==12.4.127
 - nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-nvrtc-cu12==12.4.127
 - nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cuda-runtime-cu12==12.4.127
 - nvidia-cudnn-cu12==9.10.2.21
 + nvidia-cudnn-cu12==9.1.0.70
 - nvidia-cufft-cu12==11.3.3.83
 + nvidia-cufft-cu12==11.2.1.3
 - nvidia-curand-cu12==10.3.9.90
 + nvidia-curand-cu12==10.3.5.147
 - nvidia-cusolver-cu12==11.7

!!!! **Finally, restart the kernel!** !!!!

#### Update repositories

In [112]:
!cd assignment1-basics; git pull
print()
!cd assignment2-systems; git pull

Already up to date.

remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 5 (delta 1), reused 5 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 7.17 KiB | 2.39 MiB/s, done.
From https://github.com/NiccoloSacchi/assignment2-systems
   19bfd79..92f84a5  main       -> origin/main
Updating 19bfd79..92f84a5
Fast-forward
 cs336_systems/playground.ipynb | 987 ++++++++++++++++++++++++++++++++++++++++-
 cs336_systems/prettyprint.py   |  12 +
 2 files changed, 990 insertions(+), 9 deletions(-)
 create mode 100644 cs336_systems/prettyprint.py


### GPU

#### Summaries

In [ ]:
import torch
torch.cuda.get_device_properties(0)

_CudaDeviceProperties(name='Tesla T4', major=7, minor=5, total_memory=14912MB, multi_processor_count=40, uuid=b1858ea7-262d-742f-d8fa-6c0b003fed58, L2_cache_size=4MB)

In [7]:
!nvidia-smi

Fri Mar 13 20:50:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [110]:
import torch
print(torch.cuda.memory_summary(device=None, abbreviated=False))

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  16640 KiB |  78080 KiB |    840 MiB |    824 MiB |
|       from large pool |  16640 KiB |  78080 KiB |    840 MiB |    824 MiB |
|       from small pool |      0 KiB |      2 KiB |      0 MiB |      0 MiB |
|---------------------------------------------------------------------------|
| Active memory         |  16640 KiB |  78080 KiB |    840 MiB |    824 MiB |
|       from large pool |  16640 KiB |  78080 KiB |    840 MiB |

#### Memory release

In [ ]:
from cs336_systems.prettyprint import print_gpu_memory

print_gpu_memory()

# Create a tensor on the GPU.
c = 2048
r = 1024
tensor = torch.randn(r, c, dtype=torch.float32, device="cuda")
print(f"Initialized tensor of size {r*c*32/(8*1024**2)} MB.")
print_gpu_memory()

# Delete the tensor.
del tensor
print("Deleted tensor.")

# Allocated memory is released but reserved memory is not, it will be used for
# future allocations.
print_gpu_memory()

# Force PyTorch to release all unused reserved memory back to the GPU driver.
torch.cuda.empty_cache()

# Also reserved memory is released.
print_gpu_memory()

Memory:
	Allocated: 16.2 MB
	Reserved: 40.0 MB
	Total: 14912.6875 MB
Initialized tensor of size 8.0 MB.
Memory:
	Allocated: 24.2 MB
	Reserved: 40.0 MB
	Total: 14912.6875 MB
Deleted tensor.
Memory:
	Allocated: 16.2 MB
	Reserved: 40.0 MB
	Total: 14912.6875 MB
Memory:
	Allocated: 16.2 MB
	Reserved: 40.0 MB
	Total: 14912.6875 MB


#### Arithmetics and backpropagation

In [ ]:
from cs336_systems.prettyprint import print_gpu_memory

# Check the GPU memory.
print_gpu_memory()

# Create 2 tensors and multiply.
c = 2048
r = 1024
x = torch.randn(r, c, dtype=torch.float32, requires_grad=True, device="cuda")
y = torch.randn(c, r, dtype=torch.float32, requires_grad=True, device="cuda")
print(f"Initialized 2 tensors of total size 2*{r*c*32/(8*1024**2)} MB.")
print_gpu_memory()

# Multiply them. There is an additional ~8MB. PyTorch initializes cuBLAS. This
# library loads kernels and allocates internal workspace buffers to handle the
# matrix multiplication efficiently.
z = x @ y
print(f"Multiplied the 2 tensors into a new tensor of size {r*r*32/(8*1024**2)} MB.")
print_gpu_memory()

# Sum itseld them.
k = z + z
print(f"Summed the tensor into a new tensor of size {r*r*32/(8*1024**2)} MB.")
print_gpu_memory()

# Backpropagate everything, thus populating grads.
k.sum().backward()
print("Backward propagation.")
print_gpu_memory()

# Delete all tensors and release memory.
del x, y, z, k
print("Deleted all tensors.")
torch.cuda.empty_cache()
print_gpu_memory()

Memory:
	Allocated: 16.2 MB
	Reserved: 40.0 MB
	Total: 14912.6875 MB
Initialized 2 tensors of total size 2*8.0 MB.
Memory:
	Allocated: 32.2 MB
	Reserved: 40.0 MB
	Total: 14912.6875 MB
Multiplied the 2 tensors into a new tensor of size 4.0 MB.
Memory:
	Allocated: 36.2 MB
	Reserved: 60.0 MB
	Total: 14912.6875 MB
Summed the tensor into a new tensor of size 4.0 MB.
Memory:
	Allocated: 40.2 MB
	Reserved: 60.0 MB
	Total: 14912.6875 MB
Backward propagation.
Memory:
	Allocated: 56.2 MB
	Reserved: 82.0 MB
	Total: 14912.6875 MB
Deleted all tensors.
Memory:
	Allocated: 16.2 MB
	Reserved: 40.0 MB
	Total: 14912.6875 MB


#### Intermediary results
Intermediary results are stored when grad is enabled. Here we have only 1
variable of  size 4MB, but we consume 20MB.

In [ ]:
from cs336_systems.prettyprint import print_gpu_memory

# Check the GPU memory.
print_gpu_memory()

# Create 2 tensors and multiply.
c = 2048
r = 1024
x = (
  torch.randn(r, c, dtype=torch.float32, requires_grad=True, device="cuda") @
  torch.randn(c, r, dtype=torch.float32, requires_grad=True, device="cuda")
)
print(f"Multiplied in place 2 tensors of total size 2*{r*c*32/(8*1024**2)} MB into a new tensor of size {r*r*32/(8*1024**2)} MB.")
print_gpu_memory()

# Delete all tensors and release memory.
del x
print("Deleted all tensors.")
torch.cuda.empty_cache()
print_gpu_memory()

Memory:
	Allocated: 444.4 MB
	Reserved: 476.0 MB
	Total: 14912.6875 MB
Multiplied in place 2 tensors of total size 2*8.0 MB into a new tensor of size 4.0 MB.
Memory:
	Allocated: 464.8 MB
	Reserved: 496.0 MB
	Total: 14912.6875 MB
Deleted all tensors.
Memory:
	Allocated: 444.4 MB
	Reserved: 476.0 MB
	Total: 14912.6875 MB


Intermediary results are *not* stored when requires_grad=False. Here we consume
exactly the size of the output tensor, 4MB.

In [ ]:
from cs336_systems.prettyprint import print_gpu_memory

# Check the GPU memory.
print_gpu_memory()

# Create 2 tensors and multiply.
c = 2048
r = 1024
x = (
  torch.randn(r, c, dtype=torch.float32, requires_grad=False, device="cuda") @
  torch.randn(c, r, dtype=torch.float32, requires_grad=False, device="cuda")
)
print(f"Multiplied in place 2 tensors of total size 2*{r*c*32/(8*1024**2)} MB into a new tensor of size {r*r*32/(8*1024**2)} MB.")
print_gpu_memory()

# Delete all tensors and release memory.
del x
print("Deleted all tensors.")
torch.cuda.empty_cache()
print_gpu_memory()

Memory:
	Allocated: 16.2 MB
	Reserved: 40.0 MB
	Total: 14912.6875 MB
Multiplied in place 2 tensors of total size 2*8.0 MB into a new tensor of size 4.0 MB.
Memory:
	Allocated: 20.2 MB
	Reserved: 60.0 MB
	Total: 14912.6875 MB
Deleted all tensors.
Memory:
	Allocated: 16.2 MB
	Reserved: 40.0 MB
	Total: 14912.6875 MB


#### Hard clean

In [ ]:
# Copy-paste the output to Gemini and let it figure out what variable still need
# to be deleted.
locals()

In [ ]:
# Alteratively: restart the kernel.
import os

# Kills the current process and any other python processes on the VM to make
# sure no process is using the GPU anymore.
os.system("pkill -9 python")

### Import and run model

In [1]:
# Forward pass with an untrained model.
from cs336_basics.model import TransformerLM
import torch

device=torch.device("cuda")
# device = torch.device("mps")
model = TransformerLM(
  vocab_size=10000,
  context_length=256,
  num_layers=8,
  d_model=768,
  num_heads=16,
  d_ff=1344,
  rope_theta=10000.0,
  device=device,
  dtype=torch.float32,
)

input = torch.randint(0, 10000, (1, 256), device=device)
output = model(input)
print(output.shape)

torch.Size([1, 256, 10000])


In [ ]:
!cd assignment2-systems; uv run pytest

In [ ]:
# Generate text using a trained model.
from cs336_basics.generation import generate_text
from cs336_basics.tokenizer import BPETokenizer
from cs336_basics.model import TransformerLM
from cs336_basics.io import ROOT_PATH, load_checkpoint

tok = BPETokenizer.load(ROOT_PATH / "tokenizer/tinystories_train_10000.pt")

model, _, _ = load_checkpoint(
  ROOT_PATH / "model/TinyStories/volcanic-dream-4/checkpoint_39999.pt",
  model_class=TransformerLM,
)
text = generate_text(
  modelb=model,
  tokenizer=tok,
  input_text= "Costanza",
  max_new_tokens=100,
  temperature=1.0,
  top_p=0.0,
)
print(model._init_args)
text

### ConfigsBasic Benchmark

In [3]:
# Models that we will benchmark.
from dataclasses import dataclass

@dataclass
class ModelConfig:
  vocab_size: int
  d_model: int
  d_ff: int
  num_layers: int
  num_heads: int

BATCH_SIZE = 4
MODELS = {
  "small":  ModelConfig(10000, 768, 3072, 12, 12),
  "medium": ModelConfig(10000, 1024, 4096, 24, 16),
  "large":  ModelConfig(10000, 1280, 5120, 36, 20),
  "xl":     ModelConfig(10000, 1600, 6400, 48, 25),
  "2.7B":   ModelConfig(10000, 2560, 10240, 32, 32),
}

In [ ]:
from cs336_basics.model import TransformerLM
from timeit import default_timer as timer
import numpy as np
import torch

def benchmarking_script(
  model_config: ModelConfig,
  context_length: int,
  measure_also_backward: bool,
  synchronize: bool = True,
  warmup_steps: int = 5,
  measure_steps: int = 10,
  batch_size: int = 4,
  device: torch.device = torch.device("cuda"),
) -> (int, int):
  print("Creating dummy data and model...")
  input = torch.randint(0, model_config.vocab_size, (batch_size, context_length), device=device)
  model = TransformerLM(
    vocab_size=model_config.vocab_size,
    context_length=context_length,
    num_layers=model_config.num_layers,
    d_model=model_config.d_model,
    num_heads=model_config.num_heads,
    d_ff=model_config.d_ff,
    rope_theta=10000.0,
    device=device,
    dtype=torch.float32,
  )
  
  print("Warming up...")
  for _ in range(warmup_steps):
    output = model(input)
    if measure_also_backward:
      output.sum().backward()

  print("Measuring...")
  measures_s = []
  for _ in range(measure_steps):
    # Clear gradients manually to measure 'clean' write speed (avoid readiing
    # and summing gradients from the previous step).
    for p in model.parameters():
      p.grad = None
        
    # Make sure all operations have finished before starting to measure.
    torch.cuda.synchronize()

    start = timer()
    output = model(input)
    if measure_also_backward:
      output.sum().backward()
    if synchronize:
      torch.cuda.synchronize()
    end = timer()

    measures_s.append((end - start))
  return np.mean(measures_s), np.std(measures_s)

In [13]:
benchmarking_script(
  model_config = MODELS["medium"],
  context_length = 256,
  measure_also_backward = False,
  synchronize = False,
)

Creating dummy data and model...
Warming up...
Measuring...


(np.float64(0.0784289769998395), np.float64(0.0006769804403450509))

In [14]:
benchmarking_script(
  model_config = MODELS["medium"],
  context_length = 256,
  measure_also_backward = False,
  synchronize = True,
)

Creating dummy data and model...
Warming up...
Measuring...


(np.float64(0.3672326186000646), np.float64(0.0029528172461886788))

In [12]:
benchmarking_script(
  model_config = MODELS["medium"],
  context_length = 256,
  measure_also_backward = True,
  synchronize = True,
)

Creating dummy data and model...
Warming up...
Measuring...


(np.float64(1.154933208700004), np.float64(0.019540667115277983))